In [7]:
import time
import requests
import pandas as pd

from bs4 import BeautifulSoup
from datetime import date
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

print("All imports successful!")

All imports successful!


In [8]:
# ==================================================
# 1. URL
# ==================================================

url = "https://merolagani.com/Floorsheet.aspx"


# ==================================================
# 2. SETUP CHROME
# ==================================================

chrome_options = Options()

chrome_options.add_argument(
    "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/151.0.0.0 Safari/537.36"
)

driver = webdriver.Chrome(options=chrome_options)

driver.get(url)


# ==================================================
# 3. WAIT FOR TABLE
# ==================================================

wait = WebDriverWait(driver, 30)

wait.until(
    EC.presence_of_element_located(
        (By.CSS_SELECTOR, "table.table")
    )
)

print("Website loaded successfully")


# ==================================================
# 4. FUNCTION TO EXTRACT CURRENT PAGE
# ==================================================

def extract_current_page(driver):

    # Get current HTML
    html = driver.page_source

    # Parse HTML
    soup = BeautifulSoup(html, "html.parser")

    # Find floorsheet table
    table = soup.find(
        "table",
        class_="table table-bordered table-striped table-hover sortable"
    )

    if table is None:
        print("Table not found")
        return pd.DataFrame()

    # Find table rows
    tbody = table.find("tbody")

    if tbody is None:
        print("Table body not found")
        return pd.DataFrame()

    rows = tbody.find_all("tr")

    data = []

    for row in rows:

        cells = row.find_all("td")

        if len(cells) == 8:

            data.append({
                "transaction_no": cells[1].get_text(strip=True),
                "symbol": cells[2].get_text(strip=True),
                "buyer": cells[3].get_text(strip=True),
                "seller": cells[4].get_text(strip=True),
                "quantity": cells[5].get_text(strip=True),
                "rate": cells[6].get_text(strip=True),
                "amount": cells[7].get_text(strip=True)
            })

    return pd.DataFrame(data)


# ==================================================
# 5. SCRAPE ALL PAGES
# ==================================================

all_pages = []

total_pages = 92

for page_number in range(1, total_pages + 1):

    print(
        f"Scraping page {page_number}/{total_pages}..."
    )

    # ----------------------------------------------
    # Extract current page
    # ----------------------------------------------

    df_page = extract_current_page(driver)

    print(
        f"Rows extracted: {len(df_page)}"
    )

    all_pages.append(df_page)

    # ----------------------------------------------
    # Stop after last page
    # ----------------------------------------------

    if page_number == total_pages:
        break

    # ----------------------------------------------
    # Get first transaction of current page
    # ----------------------------------------------

    old_transaction = driver.find_element(
        By.CSS_SELECTOR,
        "table.table tbody tr td:nth-child(2)"
    ).text

    # ----------------------------------------------
    # Find Next button
    # ----------------------------------------------

    next_buttons = driver.find_elements(
        By.XPATH,
        "//a[contains(@title, 'Next Page')]"
    )

    if not next_buttons:
        print("Next button not found.")
        break

    # Use the last pagination control
    next_button = next_buttons[-1]

    # ----------------------------------------------
    # Click Next
    # ----------------------------------------------

    driver.execute_script(
        "arguments[0].click();",
        next_button
    )

    # ----------------------------------------------
    # Wait until table changes
    # ----------------------------------------------

    try:

        wait.until(
            lambda d:
            d.find_element(
                By.CSS_SELECTOR,
                "table.table tbody tr td:nth-child(2)"
            ).text != old_transaction
        )

    except Exception:

        print(
            f"Could not detect page change after page "
            f"{page_number}"
        )

        time.sleep(2)


# ==================================================
# 6. CLOSE BROWSER
# ==================================================

driver.quit()


# ==================================================
# 7. COMBINE ALL PAGES
# ==================================================

df = pd.concat(
    all_pages,
    ignore_index=True
)


# ==================================================
# 8. CONVERT NUMERIC COLUMNS
# ==================================================

df["quantity"] = pd.to_numeric(
    df["quantity"],
    errors="coerce"
)

df["rate"] = pd.to_numeric(
    df["rate"],
    errors="coerce"
)

df["amount"] = pd.to_numeric(
    df["amount"].str.replace(",", ""),
    errors="coerce"
)


# ==================================================
# 9. REMOVE DUPLICATES
# ==================================================

df = df.drop_duplicates(
    subset=["transaction_no"]
).reset_index(drop=True)


# ==================================================
# 10. RESULT
# ==================================================

print("\n" + "=" * 60)
print("SCRAPING COMPLETED")
print("=" * 60)

print("Total rows:", len(df))
print("Total columns:", len(df.columns))

print("\nDataFrame:")
print(df.head())

print("\nData types:")
print(df.dtypes)

Website loaded successfully
Scraping page 1/92...
Rows extracted: 500
Could not detect page change after page 1
Scraping page 2/92...
Rows extracted: 500
Scraping page 3/92...
Rows extracted: 500
Scraping page 4/92...
Rows extracted: 500
Scraping page 5/92...
Rows extracted: 500
Scraping page 6/92...
Rows extracted: 500
Scraping page 7/92...
Rows extracted: 500
Scraping page 8/92...
Rows extracted: 500
Scraping page 9/92...
Rows extracted: 500
Scraping page 10/92...
Rows extracted: 500
Scraping page 11/92...
Rows extracted: 500
Scraping page 12/92...
Rows extracted: 500
Scraping page 13/92...
Rows extracted: 500
Scraping page 14/92...
Rows extracted: 500
Scraping page 15/92...
Rows extracted: 500
Scraping page 16/92...
Rows extracted: 500
Scraping page 17/92...
Rows extracted: 500
Scraping page 18/92...
Rows extracted: 500
Scraping page 19/92...
Rows extracted: 500
Scraping page 20/92...
Rows extracted: 500
Scraping page 21/92...
Rows extracted: 500
Scraping page 22/92...
Rows extracte

In [9]:
df.head(10)

,transaction_no,symbol,buyer,seller,quantity,rate,amount
0,2026081705009626,AHL,58,74,28.0,419.0,11732.0
1,2026081705009614,BEDC,7,34,10.0,316.9,3169.0
2,2026081705010043,BGWT,19,47,10.0,519.0,5190.0
3,2026081705009825,BHDC,8,89,10.0,427.0,4270.0
4,2026081705010001,BHL,83,13,200.0,208.4,41680.0
5,2026081705009899,BHL,59,13,131.0,208.4,27300.4
6,2026081705009898,BHL,21,13,150.0,208.6,31290.0
7,2026081705009882,BHL,21,42,50.0,208.5,10425.0
8,2026081705009834,BHL,59,98,331.0,208.4,68980.4
9,2026081705009833,BHL,99,98,12.0,208.4,2500.8


In [10]:
# Sector-wise script lists
Bank = ["NABIL", "NIMB", "SCB", "HBL", "SBI", "EBL", "NICA", "MBL", "LSL", "KBL",
        "SBL", "SANIMA", "NMB", "PRVU", "GBIME", "CZBIL", "PCBL", "ADBL", "NBL"]

manufacturing = ["BNL", "NLO", "BNT", "UNL", "HDL", "SHIVM", "GCIL", "SONA",
                 "SARBTM", "OMPL", "SAGAR", "SAIL", "SYPNL", "RSML", "PCIL", "SOPL", "ECL"]

hotel_and_tourism = ["SHL", "TRH", "OHL", "CGH", "KDL", "CITY", "BANDIPUR", "HFIN"]

other = ["NTC", "NRIC", "NRM", "MKCL", "NWCL", "HRL", "PURE", "TTL"]

hydropower = [
    "NHPC", "BPCL", "CHCL", "AHPC", "SHPC", "RIDI", "BARUN", "API",
    "NGPL", "KKHC", "DHPL", "AKPL", "SPDL", "UMHL", "CHL", "HPPL",
    "NHDL", "RADHI", "PMHPL", "KPCL", "AKJCL", "JOSHI", "UPPER", "GHL",
    "UPCL", "MHNL", "PPCL", "HURJA", "UNHPL", "RHPL", "SJCL", "HDHPC",
    "LEC", "SSHL", "MEN", "UMRH", "GLH", "SHEL", "RURU", "MKJC",
    "SAHAS", "TPC", "SPC", "NYADI", "MBJC", "BNHC", "GVL", "BHL",
    "RFPL", "DORDI", "BHDC", "HHL", "UHEWA", "SGHC", "MHL", "USHEC",
    "RHGCL", "SPHL", "PPL", "SIKLES", "EHPL", "PHCL", "BHPL", "SMHL",
    "SPL", "SMH", "MKHC", "AHL", "TAMOR", "MHCL", "SMJC", "MAKAR",
    "MKHL", "DOLTI", "BEDC", "MCHL", "IHL", "MEL", "RAWA", "USHL",
    "TSHL", "KBSH", "MEHL", "ULHC", "MANDU", "BGWT", "MSHL", "MMKJL",
    "TVCL", "VLUCL", "CKHL", "SANVI", "BHCL", "HIMSTAR", "MABEL", "DHEL",
    "BUNGAL", "SOHL", "BJHL", "SKHL", "RLEL", "SKHEL", "SIPD", "KHPL",
    "APHL", "YMHL", "TPKHL", "SNORL", "SGHL", "KAHL", "MEPDL"
]

trading = ["STC", "BBC"]

non_life_insurance = [
    "NICL", "RBCL", "HEI", "UAIL", "SPIL", "NIL", "PRIN",
    "SALICO", "IGI", "SICL", "NLG", "SGIC", "NMIC"
]

development_bank = [
    "NABBC", "EDBL", "LBBL", "MDB", "MLBL", "GBBL", "JBBL", "CORBL",
    "KSBBL", "SADBL", "SHINE", "MNBBL", "SINDU", "GRDBL", "SAPDBL", "SABBL"
]

finance = [
    "NFS", "GUFL", "BFC", "GFCL", "SIFC", "CFCL", "JFL",
    "GMFIL", "ICFC", "PROFL", "MPFL", "MFIL", "RLFL"
]

microfinance = [
    "NUBL", "CBBL", "DDBL", "SWBBL", "NMLBBL", "FMDBL", "SLBBL", "SKBBL",
    "GBLBS", "KMCDB", "MLBBL", "LLBS", "VLBS", "HLBSL", "MATRI", "JSLBB",
    "NMBMF", "GILB", "SWMF", "MERO", "NMFBS", "RSDC", "FOWAD", "SMATA",
    "MSLB", "SMB", "USLB", "WNLB", "NADEP", "ACLBSL", "SLBSL", "ALBSL",
    "GMFBS", "GLBSL", "SMFBS", "ILBS", "NICLBSL", "SMPDA", "MLBSL", "JBLB",
    "MLBS", "NESDO", "ULBSL", "CYCL", "AVYAN", "DLBS", "SHLB", "UNLB",
    "ANLB", "SWASTIK"
]

life_insurance = [
    "NLICL", "NLIC", "LICN", "ALICL", "HLI", "SJLIC", "PMLI",
    "SRLI", "ILI", "RNLI", "SNLI", "CLI", "GMLI", "CREST"
]

investment = [
    "CIT", "HIDCL", "NRN", "NIFRA", "CHDC", "ENL", "HATHY"
]
mutual_fund = [
    "SEF", "NBF2", "SIGS2", "NICBF", "NMB50", "SFMF", "LUK", "SLCF",
    "KEF", "SBCF", "PSF", "NIBSF2", "NICSF", "RMF1", "MMF1", "NBF3",
    "NICFC", "KDBY", "GIBF1", "NSIF2", "NIBLGF", "SAGF", "SFEF", "PRSF",
    "RMF2", "SIGS3", "C30MF", "LVF2", "H8020", "NICGF2", "KSY", "NIBLSTF",
    "MNMF1", "GSY", "NMBHF2", "MBLEF", "RSY", "GBIMESY2", "HLICF", "RBBF40",
    "CSY", "NSY", "SEF2", "SAEF2", "LSH12", "RSY2"
]

preference_share = [
    "NABILPNP",
    "KSBBLPNP",
    "SBLPNP",
    "NMBPNP",
    "SANIMAPNP",
    "MBLPNP"
]

# Create sector mapping
sector_mapping = {}

for script in Bank:
    sector_mapping[script] = "Banking"

for script in manufacturing:
    sector_mapping[script] = "Manufacturing"

for script in hotel_and_tourism:
    sector_mapping[script] = "Hotel and Tourism"

for script in other:
    sector_mapping[script] = "Other"

for script in hydropower:
    sector_mapping[script] = "Hydropower"

for script in trading:
    sector_mapping[script] = "Trading"

for script in non_life_insurance:
    sector_mapping[script] = "Non-Life Insurance"

for script in development_bank:
    sector_mapping[script] = "Development Bank"

for script in finance:
    sector_mapping[script] = "Finance"

for script in microfinance:
    sector_mapping[script] = "Microfinance"

for script in life_insurance:
    sector_mapping[script] = "Life Insurance"

for script in investment:
    sector_mapping[script] = "Investment"

for script in mutual_fund:
    sector_mapping[script] = "Mutual Fund"

for script in preference_share:
    sector_mapping[script] = "Preference Share"

# Create sector column
df["sector"] = df["symbol"].map(sector_mapping)

In [11]:
df["date"] = pd.Timestamp.today().date()

In [12]:
from pathlib import Path
from datetime import date

# Go from src/scraper -> project root
project_root = Path.cwd().parents[1]

# data/raw
raw_folder = project_root / "data" / "raw"/"floorsheet_data"

# Create folder if it doesn't exist
raw_folder.mkdir(parents=True, exist_ok=True)

# Filename
today_date = date.today().strftime("%Y-%m-%d")
file_path = raw_folder / f"{today_date}-stock_data.csv"

# Save
df.to_csv(file_path, index=False)

print(f"Saved successfully: {file_path}")

Saved successfully: e:\nepse-market-report\data\raw\floorsheet_data\2026-08-17-stock_data.csv


In [14]:
# for combining all the data in one file, we can use the following code:
# ============================================================
# 1. COPY SCRAPED DATA
# ============================================================

df2 = df.copy()


# ============================================================
# 2. ADD TODAY'S DATE
# ============================================================




# ============================================================
# 3. GO FROM src/scraper -> PROJECT ROOT
# ============================================================

project_root = Path.cwd().parents[1]


# ============================================================
# 4. DATA/RAW/HISTORIC_STOCK_DATA FOLDER
# ============================================================

raw_folder = (
    project_root
    / "data"
    / "raw"
    / "historic_stock_data"
)

raw_folder.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# 5. CSV FILE PATH
# ============================================================

csv_file = (
    raw_folder
    / "historic_floorsheet_data.csv"
)


# ============================================================
# 6. APPEND NEW DATA TO OLD DATA
# ============================================================

if csv_file.exists():

    # Read existing historical data
    old_df = pd.read_csv(csv_file)

    # Combine old + today's data
    historic_df = pd.concat(
        [old_df, df2],
        ignore_index=True
    )

else:

    # First time creating the file
    historic_df = df2


# ============================================================
# 7. SAVE HISTORICAL DATA
# ============================================================

historic_df.to_csv(
    csv_file,
    index=False
)


# ============================================================
# 8. INFORMATION
# ============================================================

print("Historical data saved successfully!")
print("File:", csv_file)
print("Shape:", historic_df.shape)
print("Today's date:", today)

Historical data saved successfully!
File: e:\nepse-market-report\data\raw\historic_stock_data\historic_floorsheet_data.csv
Shape: (91996, 10)
Today's date: 2026-08-17
